In [1]:
import time
import pandas as pd
import numpy as np
from tqdm import tqdm
from geopy.geocoders import Nominatim
from math import radians, sin, cos, sqrt, atan2

from src.parameters import EMISSION_FACTORS_TONNE_KM, AVERAGE_TEAM_FREIGHT_TONNES

/Users/Patron/Documents/cs524/.gamspy_venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [2]:
tqdm.pandas()

In [3]:
geolocator = Nominatim(user_agent="f1_calendar_optimization")
def get_coordinates(row, locator='city'):
    city = row[locator]
    country = row['country']
    lat_long = [None, None]

    try:
        location = geolocator.geocode(f"{city}, {country}")
        if location:
            lat_long = [location.latitude, location.longitude]
    except Exception as e:
        print(f"Error geocoding {city}, {country}: {e}")

    time.sleep(1)  # To respect Nominatim's usage policy
    return pd.Series(lat_long)
    

In [4]:
def compute_haversine(df):
    
    R = 6371  # Earth radius in km

    lat1 = np.radians(df['latitude_from'].values)
    lon1 = np.radians(df['longitude_from'].values)
    lat2 = np.radians(df['latitude_to'].values)
    lon2 = np.radians(df['longitude_to'].values)

    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1 - a))

    return R * c


In [5]:
circuits_df = pd.read_csv('data/raw/circuits.csv')

# Get latitude and longitude for each circuit
circuits_df[['latitude', 'longitude']] = circuits_df.progress_apply(get_coordinates, axis=1)
circuits_df.to_csv('data/processed/circuits.csv', index=False) 


# Compute pairwise distances between circuits
distance_df = pd.merge(circuits_df, circuits_df, how='cross', suffixes=('_from', '_to'))
distance_df = distance_df[distance_df['circuit_id_from'] < distance_df['circuit_id_to']]

# Compute Haversine distance
distance_df['distance_km'] = compute_haversine(distance_df)
print("✅ Computed Haversine distances between circuits.")

# Adjust distances for land travel in Europe
distance_df.loc[
    (distance_df['continent_from'] == 'Europe') & 
    (distance_df['continent_to']   == 'Europe'),
    'distance_km'
] *= 1.2  # Adjust for road travel in Europe


# Determine transport mode based on continents
distance_df['transport_mode'] = np.where(
    (distance_df['continent_from'] == 'Europe') & 
    (distance_df['continent_to']   == 'Europe'), "land", "air"
)


# Compute emissions (kg CO2e) for each route
distance_df['emissions_kgCO2e'] = (
    distance_df['distance_km']
    * AVERAGE_TEAM_FREIGHT_TONNES
    * distance_df['transport_mode'].map(EMISSION_FACTORS_TONNE_KM)
)

# Tidying up and saving the results
distance_df = distance_df[['circuit_id_from', 'circuit_id_to', 'distance_km', 'transport_mode', 'emissions_kgCO2e']]
distance_df.columns = ['from', 'to', 'distance_km', 'transport_mode', 'emissions_kgCO2e']
display(distance_df.head())


100%|██████████| 24/24 [00:37<00:00,  1.57s/it]

✅ Computed Haversine distances between circuits.


,from,to,distance_km,transport_mode,emissions_kgCO2e
1,BAH,SAU,1256.117695,air,8488.843381
3,BAH,JPN,8056.239086,air,54444.063745
4,BAH,CHN,6830.500467,air,46160.522154
5,BAH,MIA,12196.817405,air,82426.092021
6,BAH,MON,4332.677394,air,29280.233829


In [7]:
hq_df = pd.read_csv('data/raw/hq.csv')
hq_df[['latitude', 'longitude']] = hq_df.progress_apply(lambda row: get_coordinates(row, locator='airport_city'), axis=1)
hq_df.to_csv('data/processed/hq.csv', index=False) 
print("✅ Saved data/processed/hq.csv")
display(hq_df.head())

100%|██████████| 11/11 [00:18<00:00,  1.64s/it]

✅ Saved data/processed/hq.csv


,team_id,team_name,continent,country,city,airport_city,latitude,longitude
0,MER,Mercedes AMG F1,Europe,United Kingdom,Brackley,Birmingham,52.479699,-1.902691
1,RBR,Red Bull Racing,Europe,United Kingdom,Milton Keynes,London,51.507446,-0.127765
2,FER,Ferrari,Europe,Italy,Maranello,Bologna,44.493820,11.342633
3,MCL,McLaren,Europe,United Kingdom,Woking,London,51.507446,-0.127765
4,AMR,Aston Martin,Europe,United Kingdom,Silverstone,Birmingham,52.479699,-1.902691


In [ ]:
# Compute pairwise distances between circuits
hq_merge_df = pd.merge(hq_df, circuits_df, how='cross', suffixes=('_from', '_to'))

# Compute Haversine distance
hq_merge_df['distance_km'] = compute_haversine(hq_merge_df)
print("✅ Computed Haversine distances between circuits and HQs.")

# Adjust distances for land travel in Europe
hq_merge_df.loc[
    (hq_merge_df['continent_from'] == 'Europe') & 
    (hq_merge_df['continent_to']   == 'Europe'),
    'distance_km'
] *= 1.2  # Adjust for road travel in Europe


# Determine transport mode based on continents
hq_merge_df['transport_mode'] = np.where(
    (hq_merge_df['continent_from'] == 'Europe') & 
    (hq_merge_df['continent_to']   == 'Europe'), "land", "air"
)


# Compute emissions (kg CO2e) for each route
hq_merge_df['emissions_kgCO2e'] = (
    hq_merge_df['distance_km']
    * AVERAGE_TEAM_FREIGHT_TONNES
    * hq_merge_df['transport_mode'].map(EMISSION_FACTORS_TONNE_KM)
)

# Tidying up and saving the results
hq_merge_df = hq_merge_df[['team_id', 'circuit_id', 'distance_km', 'transport_mode', 'emissions_kgCO2e']]
hq_merge_df.columns = ['from', 'to', 'distance_km', 'transport_mode', 'emissions_kgCO2e']


distance_df = pd.concat([distance_df, hq_merge_df], ignore_index=True)
distance_df.to_csv('data/processed/distances.csv', index=False) 
print("✅ Saved data/processed/distances.csv")
display(distance_df.head())



✅ Computed Haversine distances between circuits and HQs.
✅ Saved data/processed/distances.csv


,from,to,distance_km,transport_mode,emissions_kgCO2e
0,BAH,SAU,1256.117695,air,8488.843381
1,BAH,JPN,8056.239086,air,54444.063745
2,BAH,CHN,6830.500467,air,46160.522154
3,BAH,MIA,12196.817405,air,82426.092021
4,BAH,MON,4332.677394,air,29280.233829


In [9]:
def generate_sundays(year=2026):
    """
    Generate all Sundays of a given year.
    Returns a DataFrame with date and week number.
    """
    dates = pd.date_range(start=f'{year}-01-01', end=f'{year}-12-31', freq='W-SUN')
    df = pd.DataFrame({
        'race_date': dates,
        'week_num': dates.isocalendar().week,
        'month': dates.month
    })
    return df

# Generate and save
sundays_df = generate_sundays(2026)
sundays_df.to_csv('data/processed/sundays_2026.csv', index=False)

print("✅ Created data/processed/sundays_2026.csv")
display(sundays_df.head())


✅ Created data/processed/sundays_2026.csv


,race_date,week_num,month
2026-01-04,2026-01-04,1,1
2026-01-11,2026-01-11,2,1
2026-01-18,2026-01-18,3,1
2026-01-25,2026-01-25,4,1
2026-02-01,2026-02-01,5,2
